# Prueba Banco Prompts

## Paso 1. Configuración del entorno y rutas del proyecto
**Objetivo:**
 Configurar el entorno de ejecución del notebook, identificar la raíz del proyecto y cargar las variables de entorno necesarias para acceder a la API de OpenRouter. Esto permite utilizar los módulos de src/ y mantener la configuración centralizada en el archivo .env.

In [1]:
from pathlib import Path
import sys

# Identificar la raíz del proyecto
DIRECTORIO_ACTUAL = Path.cwd().resolve()

if (DIRECTORIO_ACTUAL / "src").is_dir():
    RAIZ_PROYECTO = DIRECTORIO_ACTUAL
elif (DIRECTORIO_ACTUAL.parent / "src").is_dir():
    RAIZ_PROYECTO = DIRECTORIO_ACTUAL.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta src/. "
        "Abre el notebook desde la raíz del proyecto o desde notebooks/."
    )

# Agregar la raíz del proyecto a la ruta de búsqueda de Python
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

# Cargar las variables de entorno desde el .env de la raíz
from dotenv import load_dotenv

load_dotenv(RAIZ_PROYECTO / ".env")

print("Raíz del proyecto:", RAIZ_PROYECTO)
print("Configuración inicial correcta.")

Raíz del proyecto: C:\Users\marely\OneDrive\Documentos\PROYECTOS\2_Plantilla_Repositorio_Base\plantilla-repositorio-base
Configuración inicial correcta.


## Paso 2. Importación de módulos y banco de prompts
**Objetivo:**
Importar las funciones que desarrollamos en src/ y cargar el banco de prompts que utilizaremos para las pruebas.

In [2]:
# Importar funciones para consumir la API de OpenRouter
from src.consumir_openrouter import (
    armar_peticion,
    enviar_peticion,
    extraer_texto
)

# Importar función para interpretar y validar respuestas JSON
from src.parsear_json import parsear_respuesta_json

# Importar el banco de prompts
from src.banco_prompts import BANCO_DE_PROMPTS

# Confirmar que los módulos se cargaron correctamente
print("Módulos importados correctamente.")
print(f"Cantidad de prompts disponibles: {len(BANCO_DE_PROMPTS)}")

Módulos importados correctamente.
Cantidad de prompts disponibles: 3


## Paso 3. Exploración del banco de prompts
**Objetivo:**
Inspeccionar cómo están definidos los tres prompts y comprobar que cada uno contiene los elementos necesarios para ejecutarse y validar sus respuestas.

### 3.1 Inspeccionamos tipo de estructura

In [3]:
print("Tipo de estructura:", type(BANCO_DE_PROMPTS))
print("Cantidad de prompts:", len(BANCO_DE_PROMPTS))
print("\nPrimer elemento:")
print(BANCO_DE_PROMPTS[0])

Tipo de estructura: <class 'list'>
Cantidad de prompts: 3

Primer elemento:
{'nombre': 'clasificar_comentario', 'system_instruction': '\nEres un asistente especializado en analizar encuestas de satisfacción\nde cursos de capacitación.\n\nClasifica los aspectos mencionados en el comentario de un alumno.\n\nPara cada aspecto, identifica:\n- dimension: la dimensión de la capacitación evaluada.\n- valoracion: la valoración expresada por el alumno.\n\nDimensiones permitidas:\nfacilitacion, contenidos, materiales, infraestructura,\ncoordinacion, tiempo_practico, alimentacion, general.\n\nValoraciones permitidas:\npositiva, negativa, mixta, neutral.\n\nReglas:\n- Un comentario puede mencionar varias dimensiones.\n- Separa los aspectos positivos y negativos cuando corresponda.\n- No inventes información.\n- Si el comentario es ambiguo, utiliza general y neutral.\n- Devuelve únicamente un objeto JSON válido.\n', 'ejemplo_usuario': 'Me gustaron los contenidos, pero faltó tiempo para realizar los

### 3.2 Exploración de los prompts disponibles

In [4]:
# Explorar los prompts disponibles en el banco

for prompt in BANCO_DE_PROMPTS:
    print("=" * 60)
    print(f"Nombre del prompt: {prompt['nombre']}")
    print(f"Claves esperadas: {prompt['claves_esperadas']}")
    print(
        "Instrucción de sistema definida:",
        bool(prompt['system_instruction'])
    )
    print(
        "Ejemplo de usuario definido:",
        bool(prompt['ejemplo_usuario'])
    )
    print(
        "Esquema de respuesta definido:",
        bool(prompt['response_schema'])
    )

Nombre del prompt: clasificar_comentario
Claves esperadas: ['comentarios']
Instrucción de sistema definida: True
Ejemplo de usuario definido: True
Esquema de respuesta definido: True
Nombre del prompt: identificar_oportunidades
Claves esperadas: ['fortalezas', 'oportunidades']
Instrucción de sistema definida: True
Ejemplo de usuario definido: True
Esquema de respuesta definido: True
Nombre del prompt: generar_recomendaciones
Claves esperadas: ['recomendaciones']
Instrucción de sistema definida: True
Ejemplo de usuario definido: True
Esquema de respuesta definido: True


### 3.3 Verificar las claves esperadas y los esquemas

In [5]:
#  Verificar coherencia entre claves y esquema JSON

for prompt in BANCO_DE_PROMPTS:
    claves_esperadas = set(prompt["claves_esperadas"])
    propiedades_schema = set(
        prompt["response_schema"].get("properties", {}).keys()
    )

    coherentes = claves_esperadas.issubset(propiedades_schema)

    print(f"\nPrompt: {prompt['nombre']}")
    print(f"Claves esperadas: {claves_esperadas}")
    print(f"Propiedades del esquema: {propiedades_schema}")
    print(f"¿Son coherentes?: {coherentes}")


Prompt: clasificar_comentario
Claves esperadas: {'comentarios'}
Propiedades del esquema: {'comentarios'}
¿Son coherentes?: True

Prompt: identificar_oportunidades
Claves esperadas: {'fortalezas', 'oportunidades'}
Propiedades del esquema: {'fortalezas', 'oportunidades'}
¿Son coherentes?: True

Prompt: generar_recomendaciones
Claves esperadas: {'recomendaciones'}
Propiedades del esquema: {'recomendaciones'}
¿Son coherentes?: True


## Paso 4. Prueba individual de un prompt
**Objetivo:** Ejecutar una prueba individual del prompt clasificar_comentario utilizando un comentario de una encuesta de satisfacción. Se preparará y enviará una solicitud a OpenRouter, incorporando las instrucciones de sistema y el esquema JSON definido, para comprobar si el modelo identifica correctamente las dimensiones evaluadas y sus valoraciones.

### 4.1 Seleccionar el prompt

In [6]:
# Paso 4.1. Seleccionar el prompt que vamos a probar

prompt_prueba = next(
    prompt
    for prompt in BANCO_DE_PROMPTS
    if prompt["nombre"] == "clasificar_comentario"
)

print("Prompt seleccionado:", prompt_prueba["nombre"])
print("\nComentario de prueba:")
print(prompt_prueba["ejemplo_usuario"])

Prompt seleccionado: clasificar_comentario

Comentario de prueba:
Me gustaron los contenidos, pero faltó tiempo para realizar los ejercicios.


### 4.2 Preparar la petición de Openrouter

In [7]:
# Preparar la petición a OpenRouter

url, headers, body = armar_peticion(
    pregunta=prompt_prueba["ejemplo_usuario"],
    system_instruction=prompt_prueba["system_instruction"],
    response_schema=prompt_prueba["response_schema"]
)

print("URL de la petición:", url)
print("Modelo seleccionado:", body["model"])
print("Cantidad de mensajes:", len(body["messages"]))
print("Formato de respuesta:", body.get("response_format", {}).get("type"))
print("\nPetición preparada correctamente.")

URL de la petición: https://openrouter.ai/api/v1/chat/completions
Modelo seleccionado: openai/gpt-4o-mini
Cantidad de mensajes: 2
Formato de respuesta: json_schema

Petición preparada correctamente.


### 4.3 Enviar la petición y obtener la respuesta

In [8]:
# Enviar la petición a OpenRouter

datos_respuesta = enviar_peticion(url, headers, body)

print("Petición enviada correctamente.")
print("Respuesta recibida desde OpenRouter.")

Petición enviada correctamente.
Respuesta recibida desde OpenRouter.


### 4.4 Extraer el texto generado

In [9]:
#  Extraer el texto generado por el modelo

texto_generado = extraer_texto(datos_respuesta)

print("Respuesta del modelo:\n")
print(texto_generado)

Respuesta del modelo:

{"comentarios":[{"dimension":"contenidos","valoracion":"positiva"},{"dimension":"tiempo_practico","valoracion":"negativa"}]}


## Paso 5. Validación de la respuesta JSON
**Objetivo:** Comprobar que la respuesta generada por el modelo tenga un formato JSON válido y contenga las claves esperadas por el prompt. Esta validación permite detectar errores de formato o respuestas incompletas antes de utilizar los resultados en etapas posteriores del análisis.

### 5.1 Convertir y validar el JSON

In [10]:
# Paso 5.1. Validar la respuesta JSON

resultado = parsear_respuesta_json(
    texto_generado,
    claves_esperadas=prompt_prueba["claves_esperadas"]
)

print("JSON válido.")
print("Claves esperadas presentes.")
print("\nResultado convertido a Python:")
print(resultado)

JSON válido.
Claves esperadas presentes.

Resultado convertido a Python:
{'comentarios': [{'dimension': 'contenidos', 'valoracion': 'positiva'}, {'dimension': 'tiempo_practico', 'valoracion': 'negativa'}]}


## Paso 6. Ejecución de pruebas para los tres prompts
**Objetivo:** Evaluar el funcionamiento de los tres prompts del banco mediante solicitudes a OpenRouter, comprobando si las respuestas generadas son JSON válidos y contienen las claves esperadas. Los resultados permitirán identificar posibles errores y comparar el comportamiento de los prompts antes de integrarlos en la aplicación.

In [11]:
# Paso 6.1. Preparar la ejecución de las pruebas

resultados_pruebas = []

print(f"Prompts que se evaluarán: {len(BANCO_DE_PROMPTS)}")
print("\nPrompts incluidos:")

for prompt in BANCO_DE_PROMPTS:
    print(f"- {prompt['nombre']}")

print("\nEstructura de registro preparada.")

Prompts que se evaluarán: 3

Prompts incluidos:
- clasificar_comentario
- identificar_oportunidades
- generar_recomendaciones

Estructura de registro preparada.


In [12]:

# Paso 6.2. Ejecutar los prompts del banco

for prompt in BANCO_DE_PROMPTS:
    print(f"\n{'=' * 50}")
    print(f"Evaluando: {prompt['nombre']}")
    print("=" * 50)

    try:
        # 1. Preparar la petición con los datos del prompt.
        url, headers, body = armar_peticion(
            pregunta=prompt["ejemplo_usuario"],
            system_instruction=prompt["system_instruction"],
            response_schema=prompt["response_schema"]
        )

        # 2. Enviar la petición a OpenRouter.
        datos_respuesta = enviar_peticion(url, headers, body)

        # 3. Extraer el texto generado por el modelo.
        texto = extraer_texto(datos_respuesta)

        # 4. Validar el JSON y las claves esperadas.
        resultado = parsear_respuesta_json(
            texto,
            claves_esperadas=prompt["claves_esperadas"]
        )

        # 5. Registrar el resultado de la prueba.
        resultados_pruebas.append({
            "nombre": prompt["nombre"],
            "estado": "correcto",
            "respuesta": resultado,
            "error": None
        })

        print("Estado: correcto")
        print("Respuesta:")
        print(resultado)

    except Exception as error:
        # Registrar el error y continuar con el siguiente prompt.
        resultados_pruebas.append({
            "nombre": prompt["nombre"],
            "estado": "error",
            "respuesta": None,
            "error": str(error)
        })

        print("Estado: error")
        print(f"Detalle: {error}")

print("\nPruebas finalizadas.")



Evaluando: clasificar_comentario
Estado: correcto
Respuesta:
{'comentarios': [{'dimension': 'contenidos', 'valoracion': 'positiva'}, {'dimension': 'tiempo_practico', 'valoracion': 'negativa'}]}

Evaluando: identificar_oportunidades
Estado: correcto
Respuesta:
{'fortalezas': ['La facilitadora explicó claramente los contenidos.', 'Las explicaciones fueron claras.'], 'oportunidades': ['Me habría gustado disponer de más tiempo para practicar.', 'Faltaron ejercicios.']}

Evaluando: generar_recomendaciones
Estado: correcto
Respuesta:
{'recomendaciones': [{'dimension': 'tiempo_practico', 'accion': 'Aumentar la duración de las sesiones de capacitación para incluir al menos un 30% más de tiempo destinado a ejercicios prácticos y dinámicas interactivas.', 'prioridad': 'alta'}, {'dimension': 'contenidos', 'accion': 'Revisar y actualizar los materiales de capacitación para incluir más ejemplos específicos que reflejen situaciones reales y cotidianas del entorno laboral de los participantes.', 'pr

In [13]:
# Paso 6.4. Resumen de resultados

total_pruebas = len(resultados_pruebas)

pruebas_correctas = sum(
    1 for prueba in resultados_pruebas
    if prueba["estado"] == "correcto"
)

pruebas_fallidas = total_pruebas - pruebas_correctas

porcentaje_exito = (
    pruebas_correctas / total_pruebas * 100
    if total_pruebas > 0
    else 0
)

print("RESUMEN DE LAS PRUEBAS")
print("-" * 35)
print(f"Total de pruebas: {total_pruebas}")
print(f"Pruebas correctas: {pruebas_correctas}")
print(f"Pruebas fallidas: {pruebas_fallidas}")
print(f"Porcentaje de ejecución correcta: {porcentaje_exito:.1f}%")

print("\nDetalle por prompt:")

for prueba in resultados_pruebas:
    print(f"- {prueba['nombre']}: {prueba['estado']}")


RESUMEN DE LAS PRUEBAS
-----------------------------------
Total de pruebas: 3
Pruebas correctas: 3
Pruebas fallidas: 0
Porcentaje de ejecución correcta: 100.0%

Detalle por prompt:
- clasificar_comentario: correcto
- identificar_oportunidades: correcto
- generar_recomendaciones: correcto


## Paso 7. Resumen de resultados

In [14]:
# Paso 7.1. Definir las reglas de consistencia

DIMENSIONES_PERMITIDAS = {
    "facilitacion",
    "contenidos",
    "materiales",
    "infraestructura",
    "coordinacion",
    "tiempo_practico",
    "alimentacion",
    "general"
}

VALORACIONES_PERMITIDAS = {
    "positiva",
    "negativa",
    "mixta",
    "neutral"
}

print("Dimensiones permitidas:", len(DIMENSIONES_PERMITIDAS))
print("Valoraciones permitidas:", len(VALORACIONES_PERMITIDAS))
print("\nReglas de consistencia definidas.")


Dimensiones permitidas: 8
Valoraciones permitidas: 4

Reglas de consistencia definidas.


In [15]:
# Paso 7.2. Validar las categorías y valoraciones

prueba_clasificacion = next(
    prueba for prueba in resultados_pruebas
    if prueba["nombre"] == "clasificar_comentario"
)

comentarios = prueba_clasificacion["respuesta"]["comentarios"]

errores_consistencia = []

for indice, comentario in enumerate(comentarios, start=1):

    # Comprobar que existan las claves obligatorias.
    if not all(clave in comentario for clave in ["dimension", "valoracion"]):
        errores_consistencia.append(
            f"Comentario {indice}: faltan claves obligatorias."
        )
        continue

    # Comprobar que la dimensión esté permitida.
    if comentario["dimension"] not in DIMENSIONES_PERMITIDAS:
        errores_consistencia.append(
            f"Comentario {indice}: dimensión no permitida "
            f"({comentario['dimension']})."
        )

    # Comprobar que la valoración esté permitida.
    if comentario["valoracion"] not in VALORACIONES_PERMITIDAS:
        errores_consistencia.append(
            f"Comentario {indice}: valoración no permitida "
            f"({comentario['valoracion']})."
        )

# Mostrar el resultado de la validación.
if errores_consistencia:
    print("Se encontraron inconsistencias:")
    for error in errores_consistencia:
        print("-", error)
else:
    print("Validación de consistencia superada.")
    print(f"Comentarios revisados: {len(comentarios)}")
    print("Todas las dimensiones y valoraciones están permitidas.")


Validación de consistencia superada.
Comentarios revisados: 2
Todas las dimensiones y valoraciones están permitidas.


In [16]:
# Paso 7.3. Validar fortalezas y oportunidades

prueba_oportunidades = next(
    prueba for prueba in resultados_pruebas
    if prueba["nombre"] == "identificar_oportunidades"
)

respuesta_oportunidades = prueba_oportunidades["respuesta"]

claves_requeridas = ["fortalezas", "oportunidades"]
errores_oportunidades = []

# Comprobar que existan las claves requeridas.
for clave in claves_requeridas:
    if clave not in respuesta_oportunidades:
        errores_oportunidades.append(
            f"Falta la clave obligatoria: {clave}"
        )

# Comprobar que cada valor sea una lista de textos.
for clave in claves_requeridas:
    if clave in respuesta_oportunidades:
        valor = respuesta_oportunidades[clave]

        if not isinstance(valor, list):
            errores_oportunidades.append(
                f"'{clave}' debe ser una lista."
            )
        elif not all(
            isinstance(elemento, str) and elemento.strip()
            for elemento in valor
        ):
            errores_oportunidades.append(
                f"'{clave}' debe contener únicamente textos no vacíos."
            )

# Mostrar el resultado.
if errores_oportunidades:
    print("Se encontraron inconsistencias:")
    for error in errores_oportunidades:
        print("-", error)
else:
    print("Validación de consistencia superada.")
    print(f"Fortalezas identificadas: {len(respuesta_oportunidades['fortalezas'])}")
    print(f"Oportunidades identificadas: {len(respuesta_oportunidades['oportunidades'])}")
    print("Las dos categorías contienen listas de texto válidas.")


Validación de consistencia superada.
Fortalezas identificadas: 2
Oportunidades identificadas: 2
Las dos categorías contienen listas de texto válidas.


In [ ]:

# Paso 7.4. Validar las recomendaciones

prueba_recomendaciones = next(
    prueba for prueba in resultados_pruebas
    if prueba["nombre"] == "generar_recomendaciones"
)

respuesta_recomendaciones = prueba_recomendaciones["respuesta"]

PRIORIDADES_PERMITIDAS = {"alta", "media", "baja"}

errores_recomendaciones = []

# Comprobar que exista la clave principal.
if "recomendaciones" not in respuesta_recomendaciones:
    errores_recomendaciones.append(
        "Falta la clave obligatoria: recomendaciones."
    )
else:
    recomendaciones = respuesta_recomendaciones["recomendaciones"]

    # Comprobar que las recomendaciones estén en una lista.
    if not isinstance(recomendaciones, list):
        errores_recomendaciones.append(
            "La clave 'recomendaciones' debe contener una lista."
        )
    else:
        for indice, recomendacion in enumerate(recomendaciones, start=1):

            # Comprobar que cada recomendación sea un diccionario.
            if not isinstance(recomendacion, dict):
                errores_recomendaciones.append(
                    f"Recomendación {indice}: debe ser un objeto."
                )
                continue

            # Comprobar las claves obligatorias.
            claves_requeridas = ["dimension", "accion", "prioridad"]

            for clave in claves_requeridas:
                if clave not in recomendacion:
                    errores_recomendaciones.append(
                        f"Recomendación {indice}: falta la clave '{clave}'."
                    )

            # Validar la dimensión.
            dimension = recomendacion.get("dimension")

            if dimension not in DIMENSIONES_PERMITIDAS:
                errores_recomendaciones.append(
                    f"Recomendación {indice}: dimensión no permitida "
                    f"({dimension})."
                )

            # Validar la acción.
            accion = recomendacion.get("accion")

            if not isinstance(accion, str) or not accion.strip():
                errores_recomendaciones.append(
                    f"Recomendación {indice}: la acción debe ser un texto no vacío."
                )

            # Validar la prioridad.
            prioridad = recomendacion.get("prioridad")

            if prioridad not in PRIORIDADES_PERMITIDAS:
                errores_recomendaciones.append(
                    f"Recomendación {indice}: prioridad no permitida "
                    f"({prioridad})."
                )

# Mostrar el resultado.
if errores_recomendaciones:
    print("Se encontraron inconsistencias:")

    for error in errores_recomendaciones:
        print("-", error)
else:
    print("Validación de consistencia superada.")
    errores_recomendaciones = []
    recomendaciones = []

    # Comprobar que exista la clave principal.
    if "recomendaciones" not in respuesta_recomendaciones:
        errores_recomendaciones.append(
            "Falta la clave obligatoria: recomendaciones."
        )
    else:
        recomendaciones = respuesta_recomendaciones["recomendaciones"]
    print("Dimensiones, acciones y prioridades válidas.")


Validación de consistencia superada.
Recomendaciones revisadas: 2
Dimensiones, acciones y prioridades válidas.


## Paso 8. Registro de resultados
**Objetivo:**

En esta etapa se consolidan los resultados de las validaciones realizadas sobre los tres prompts del banco.

Se revisaron las dimensiones y valoraciones permitidas, la estructura de las fortalezas y oportunidades, y los campos obligatorios de las recomendaciones.

Estas comprobaciones permiten evaluar la consistencia estructural de las respuestas. La coherencia semántica y la pertinencia de las recomendaciones requieren pruebas adicionales.
```

In [18]:

# Paso 8.1. Resumen de validaciones

validaciones = [
    {
        "prompt": "clasificar_comentario",
        "aspecto": "Dimensiones y valoraciones permitidas",
        "estado": "superada"
    },
    {
        "prompt": "identificar_oportunidades",
        "aspecto": "Claves obligatorias y listas de texto",
        "estado": "superada"
    },
    {
        "prompt": "generar_recomendaciones",
        "aspecto": "Campos, dimensiones, acciones y prioridades",
        "estado": "superada"
    }
]

total_validaciones = len(validaciones)
validaciones_superadas = sum(
    1 for validacion in validaciones
    if validacion["estado"] == "superada"
)

print("RESUMEN DE VALIDACIONES")
print("-" * 40)

for validacion in validaciones:
    print(f"\nPrompt: {validacion['prompt']}")
    print(f"Aspecto: {validacion['aspecto']}")
    print(f"Estado: {validacion['estado']}")

print("\n" + "-" * 40)
print(f"Validaciones realizadas: {total_validaciones}")
print(f"Validaciones superadas: {validaciones_superadas}")
print(
    "Porcentaje de validaciones superadas: "
    f"{validaciones_superadas / total_validaciones * 100:.1f}%"
)


RESUMEN DE VALIDACIONES
----------------------------------------

Prompt: clasificar_comentario
Aspecto: Dimensiones y valoraciones permitidas
Estado: superada

Prompt: identificar_oportunidades
Aspecto: Claves obligatorias y listas de texto
Estado: superada

Prompt: generar_recomendaciones
Aspecto: Campos, dimensiones, acciones y prioridades
Estado: superada

----------------------------------------
Validaciones realizadas: 3
Validaciones superadas: 3
Porcentaje de validaciones superadas: 100.0%


## Paso 9. Evaluación semántica

La evaluación semántica permite comprobar si las respuestas del modelo son coherentes con el significado del comentario original.

En la primera prueba, el modelo identificó correctamente una valoración positiva sobre los contenidos y una valoración negativa sobre el tiempo práctico.

El resultado es coherente con el comentario analizado. Sin embargo, esta prueba corresponde a un único ejemplo y no demuestra que el prompt clasifique correctamente todos los comentarios posibles.
```

In [19]:

# Paso 9.1. Evaluar semánticamente el primer comentario

comentario_original = (
    "Me gustaron los contenidos, pero faltó tiempo para realizar los ejercicios."
)

resultado_clasificacion = next(
    prueba["respuesta"]
    for prueba in resultados_pruebas
    if prueba["nombre"] == "clasificar_comentario"
)

print("COMENTARIO ORIGINAL")
print(comentario_original)

print("\nCLASIFICACIONES DEL MODELO")

for comentario in resultado_clasificacion["comentarios"]:
    print(
        f"- Dimensión: {comentario['dimension']} | "
        f"Valoración: {comentario['valoracion']}"
    )

# Evaluación semántica manual basada en el comentario original.
evaluacion_semantica = {
    "contenidos": "positiva",
    "tiempo_practico": "negativa"
}

clasificaciones_correctas = all(
    evaluacion_semantica.get(comentario["dimension"])
    == comentario["valoracion"]
    for comentario in resultado_clasificacion["comentarios"]
)

print("\nRESULTADO DE LA EVALUACIÓN SEMÁNTICA")

if clasificaciones_correctas:
    print("Las clasificaciones coinciden con la evaluación de referencia.")
else:
    print("Se detectaron diferencias respecto de la evaluación de referencia.")


COMENTARIO ORIGINAL
Me gustaron los contenidos, pero faltó tiempo para realizar los ejercicios.

CLASIFICACIONES DEL MODELO
- Dimensión: contenidos | Valoración: positiva
- Dimensión: tiempo_practico | Valoración: negativa

RESULTADO DE LA EVALUACIÓN SEMÁNTICA
Las clasificaciones coinciden con la evaluación de referencia.


In [20]:
# Paso 9.2. Preparar la prueba con un comentario ambiguo

comentario_ambiguo = (
    "La capacitación estuvo bien, pero esperaba algo diferente."
)

url_ambigua, headers_ambiguos, body_ambiguo = armar_peticion(
    pregunta=comentario_ambiguo,
    system_instruction=prompt_prueba["system_instruction"],
    response_schema=prompt_prueba["response_schema"]
)

print("Comentario de prueba:")
print(comentario_ambiguo)

print("\nModelo seleccionado:", body_ambiguo["model"])
print("Petición preparada correctamente.")

Comentario de prueba:
La capacitación estuvo bien, pero esperaba algo diferente.

Modelo seleccionado: openai/gpt-4o-mini
Petición preparada correctamente.


In [21]:
# Paso 9.3. Ejecutar la prueba con el comentario ambiguo

datos_ambiguos = enviar_peticion(
    url_ambigua,
    headers_ambiguos,
    body_ambiguo
)

texto_ambiguo = extraer_texto(datos_ambiguos)

print("Respuesta del modelo:\n")
print(texto_ambiguo)

Respuesta del modelo:

{"comentarios":[{"dimension":"general","valoracion":"neutral"}]}


```markdown
### Paso 9.4. Evaluación semántica del comentario ambiguo

**Comentario analizado:**

"La capacitación estuvo bien, pero esperaba algo diferente".

**Respuesta del modelo:**
- Dimensión: `general`
- Valoración: `neutral`

**Evaluación:**

La dimensión `general` es razonable, ya que el comentario no identifica un aspecto específico de la capacitación.

La valoración `neutral` es discutible. La expresión "estuvo bien" sugiere una apreciación favorable, mientras que "esperaba algo diferente" puede indicar una expectativa no satisfecha.

**Conclusión:**

La respuesta cumple las restricciones estructurales y utiliza valores permitidos, pero su interpretación semántica es discutible. Este resultado muestra que la validez estructural no garantiza una clasificación semánticamente adecuada.
```

In [22]:

# Paso 9.4. Validar y registrar la respuesta ambigua

resultado_ambiguo = parsear_respuesta_json(
    texto_ambiguo,
    claves_esperadas=prompt_prueba["claves_esperadas"]
)

comentarios_ambiguos = resultado_ambiguo["comentarios"]

errores_ambiguos = []

for indice, comentario in enumerate(comentarios_ambiguos, start=1):

    # Comprobar que cada elemento sea un objeto.
    if not isinstance(comentario, dict):
        errores_ambiguos.append(
            f"Comentario {indice}: debe ser un objeto."
        )
        continue

    # Comprobar que existan las claves obligatorias.
    if not all(clave in comentario for clave in ["dimension", "valoracion"]):
        errores_ambiguos.append(
            f"Comentario {indice}: faltan claves obligatorias."
        )
        continue

    # Comprobar los valores permitidos.
    if comentario["dimension"] not in DIMENSIONES_PERMITIDAS:
        errores_ambiguos.append(
            f"Comentario {indice}: dimensión no permitida."
        )

    if comentario["valoracion"] not in VALORACIONES_PERMITIDAS:
        errores_ambiguos.append(
            f"Comentario {indice}: valoración no permitida."
        )

# Mostrar los resultados de la validación.
print("Comentario evaluado:")
print(comentario_ambiguo)

print("\nRespuesta validada:")
print(resultado_ambiguo)

if errores_ambiguos:
    print("\nSe encontraron inconsistencias:")
    for error in errores_ambiguos:
        print("-", error)
else:
    print("\nValidación estructural y de valores permitidos: superada.")

# Registrar la evaluación semántica humana.
evaluacion_ambigua = {
    "dimension": "general",
    "valoracion": "neutral",
    "coherencia_semantica": "discutible",
    "observacion": (
        "La dimensión general es razonable, pero la valoración neutral "
        "es discutible porque el comentario combina una apreciación "
        "favorable con una expectativa no satisfecha."
    )
}

print("\nEvaluación semántica de referencia:")
print(evaluacion_ambigua)


Comentario evaluado:
La capacitación estuvo bien, pero esperaba algo diferente.

Respuesta validada:
{'comentarios': [{'dimension': 'general', 'valoracion': 'neutral'}]}

Validación estructural y de valores permitidos: superada.

Evaluación semántica de referencia:
{'dimension': 'general', 'valoracion': 'neutral', 'coherencia_semantica': 'discutible', 'observacion': 'La dimensión general es razonable, pero la valoración neutral es discutible porque el comentario combina una apreciación favorable con una expectativa no satisfecha.'}



## Paso 10. Consolidación de resultados

En esta etapa se consolidan las pruebas realizadas al banco de prompts, considerando tanto la ejecución de las peticiones como la validación estructural y semántica de las respuestas.

Los tres prompts se ejecutaron correctamente y sus respuestas superaron las validaciones estructurales y de valores permitidos implementadas.

En la evaluación semántica se analizaron dos comentarios:
- Un comentario explícito, cuyas clasificaciones coincidieron con la evaluación de referencia.
- Un comentario ambiguo, cuya respuesta cumplió las restricciones estructurales, pero presentó una valoración semántica discutible.

Los resultados muestran que la validez estructural no garantiza por sí sola una interpretación semántica adecuada.

Las pruebas realizadas son iniciales y no permiten garantizar el comportamiento de los prompts ante todas las entradas posibles.


In [23]:
# Paso 10.1. Resumir las validaciones estructurales

resumen_validaciones = [
    {
        "prompt": "clasificar_comentario",
        "validacion": "Dimensiones y valoraciones permitidas",
        "estado": "superada"
    },
    {
        "prompt": "identificar_oportunidades",
        "validacion": "Claves obligatorias y listas de texto",
        "estado": "superada"
    },
    {
        "prompt": "generar_recomendaciones",
        "validacion": "Campos y valores permitidos",
        "estado": "superada"
    }
]

print("RESUMEN DE VALIDACIONES ESTRUCTURALES")
print("=" * 45)

for validacion in resumen_validaciones:
    print(f"\nPrompt: {validacion['prompt']}")
    print(f"Validación: {validacion['validacion']}")
    print(f"Estado: {validacion['estado']}")

total = len(resumen_validaciones)
superadas = sum(
    1 for validacion in resumen_validaciones
    if validacion["estado"] == "superada"
)

print("\n" + "=" * 45)
print(f"Total de validaciones: {total}")
print(f"Validaciones superadas: {superadas}")
print(f"Porcentaje superado: {superadas / total * 100:.1f}%")


RESUMEN DE VALIDACIONES ESTRUCTURALES

Prompt: clasificar_comentario
Validación: Dimensiones y valoraciones permitidas
Estado: superada

Prompt: identificar_oportunidades
Validación: Claves obligatorias y listas de texto
Estado: superada

Prompt: generar_recomendaciones
Validación: Campos y valores permitidos
Estado: superada

Total de validaciones: 3
Validaciones superadas: 3
Porcentaje superado: 100.0%


In [24]:

# Paso 10.2. Generar el informe consolidado

total_prompts = len(resultados_pruebas)

ejecuciones_correctas = sum(
    1 for prueba in resultados_pruebas
    if prueba["estado"] == "correcto"
)

ejecuciones_fallidas = total_prompts - ejecuciones_correctas

print("INFORME CONSOLIDADO DE PRUEBAS")
print("=" * 45)

print("\n1. EJECUCIÓN DE LOS PROMPTS")
print(f"Total de prompts: {total_prompts}")
print(f"Ejecuciones correctas: {ejecuciones_correctas}")
print(f"Ejecuciones fallidas: {ejecuciones_fallidas}")

if total_prompts > 0:
    porcentaje = ejecuciones_correctas / total_prompts * 100
    print(f"Porcentaje de ejecución correcta: {porcentaje:.1f}%")

print("\n2. VALIDACIÓN ESTRUCTURAL")
print(f"Validaciones superadas: {superadas}/{total}")
print(f"Porcentaje de validaciones superadas: {superadas / total * 100:.1f}%")

print("\n3. EVALUACIÓN SEMÁNTICA")
print("- Comentario explícito: coherente con la referencia.")
print("- Comentario ambiguo: valoración discutible.")

print("\n4. CONCLUSIÓN")
print(
    "Los tres prompts superaron las pruebas de ejecución y las "
    "validaciones estructurales implementadas."
)
print(
    "La prueba con un comentario ambiguo mostró que una respuesta "
    "estructuralmente válida puede tener una interpretación discutible."
)
print(
    "Se necesitan más casos de prueba para evaluar la consistencia "
    "semántica de forma más amplia."
)


INFORME CONSOLIDADO DE PRUEBAS

1. EJECUCIÓN DE LOS PROMPTS
Total de prompts: 3
Ejecuciones correctas: 3
Ejecuciones fallidas: 0
Porcentaje de ejecución correcta: 100.0%

2. VALIDACIÓN ESTRUCTURAL
Validaciones superadas: 3/3
Porcentaje de validaciones superadas: 100.0%

3. EVALUACIÓN SEMÁNTICA
- Comentario explícito: coherente con la referencia.
- Comentario ambiguo: valoración discutible.

4. CONCLUSIÓN
Los tres prompts superaron las pruebas de ejecución y las validaciones estructurales implementadas.
La prueba con un comentario ambiguo mostró que una respuesta estructuralmente válida puede tener una interpretación discutible.
Se necesitan más casos de prueba para evaluar la consistencia semántica de forma más amplia.
